# Level 2 — Task 4: Unveiling the Android App Market

## Objective
Perform a detailed analysis of the Google Play Store app market to understand:
- app categories and market concentration
- ratings and reviews
- installation reach
- free vs paid apps
- pricing
- content ratings
- app size
- relationships between popularity and ratings

## Dataset
Google Play Store Apps dataset (`googleplaystore.csv`), a commonly used Kaggle dataset with approximately 10,841 raw records and 13 columns.

The notebook expects the CSV in `../data/googleplaystore.csv`.

## Workflow
1. Load and inspect the raw data.
2. Clean duplicate app records and malformed rows.
3. Convert Reviews, Installs, Size and Price to numeric values.
4. Parse dates.
5. Analyse categories, ratings, installs, reviews and prices.
6. Compare free and paid apps.
7. Study ratings vs popularity.
8. Identify top apps by installs and reviews.
9. Generate actionable market insights.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = "../data/googleplaystore.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "googleplaystore.csv was not found in ../data/. "
        "Download the Google Play Store Apps dataset and place the CSV there."
    )

raw = pd.read_csv(DATA_PATH)

print("Raw shape:", raw.shape)
display(raw.head())
print("\nColumns:")
print(raw.columns.tolist())


## 1. Initial data-quality inspection


In [ ]:
print("Data types:")
display(raw.dtypes)

print("\nMissing values:")
display(raw.isna().sum().sort_values(ascending=False))

print("\nDuplicate rows:", raw.duplicated().sum())


The original dataset contains some malformed rows and duplicated app records. Cleaning is necessary before calculating market statistics.


In [ ]:
df = raw.copy()

# Remove rows where the App column is missing
df = df.dropna(subset=["App"]).copy()

# Convert key columns
df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")
df["Price"] = (
    df["Price"].astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

df["Installs"] = (
    df["Installs"].astype(str)
    .str.replace("+", "", regex=False)
    .str.replace(",", "", regex=False)
)
df["Installs"] = pd.to_numeric(df["Installs"], errors="coerce")

# Clean size: convert k to MB; "Varies with device" becomes NaN
def size_to_mb(value):
    value = str(value).strip()
    if value.lower() in {"nan", "varies with device"}:
        return np.nan
    if value.endswith("M"):
        return float(value[:-1])
    if value.endswith("k"):
        return float(value[:-1]) / 1024
    return np.nan

df["Size_MB"] = df["Size"].apply(size_to_mb)

# Parse date
df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors="coerce")

# Rating is numeric
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

# Remove malformed category rows by keeping expected uppercase category pattern
df = df[df["Category"].astype(str).str.match(r"^[A-Z_]+$", na=False)].copy()

print("Shape after basic cleaning:", df.shape)
display(df.head())


## 2. Remove duplicate app entries

The dataset can contain multiple records for the same app. For market-level analysis, keeping the latest/first available app record is preferable to counting the same app repeatedly.


In [ ]:
before = len(df)
df = df.drop_duplicates(subset=["App"], keep="last").copy()
print("Rows removed because of duplicate App names:", before - len(df))
print("Unique apps:", df["App"].nunique())


## 3. Missing-value handling

Rating and size are allowed to remain missing because their absence can represent unavailable information. For aggregate charts, Pandas naturally excludes missing numerical values from means.


In [ ]:
quality = pd.DataFrame({
    "Column": df.columns,
    "Missing": [df[c].isna().sum() for c in df.columns],
    "Missing_%": [df[c].isna().mean()*100 for c in df.columns]
}).sort_values("Missing", ascending=False)

display(quality)


## 4. Category analysis


In [ ]:
category_counts = df["Category"].value_counts().head(15)

plt.figure(figsize=(10, 7))
sns.barplot(x=category_counts.values, y=category_counts.index)
plt.title("Top App Categories by Number of Apps")
plt.xlabel("Number of Apps")
plt.ylabel("Category")
plt.show()


**Observation:** App supply is concentrated in a limited number of categories. This indicates that developers compete more heavily in popular categories, while smaller categories may have less competition but also a smaller audience.


## 5. Average rating by category


In [ ]:
category_rating = (
    df.groupby("Category")["Rating"]
      .agg(["mean", "count"])
      .query("count >= 20")
      .sort_values("mean", ascending=False)
)

plt.figure(figsize=(10, 7))
sns.barplot(
    data=category_rating.head(15).reset_index(),
    x="mean", y="Category"
)
plt.xlim(0, 5)
plt.title("Top Categories by Average Rating (minimum 20 rated apps)")
plt.xlabel("Average Rating")
plt.ylabel("Category")
plt.show()


**Observation:** Average rating alone should not determine market opportunity. A category with a high average rating but very few apps or installs may have less commercial reach than a larger category.


## 6. Install and review analysis


In [ ]:
install_by_category = (
    df.groupby("Category")["Installs"]
      .sum()
      .sort_values(ascending=False)
      .head(15)
)

plt.figure(figsize=(10, 7))
sns.barplot(x=install_by_category.values, y=install_by_category.index)
plt.title("Top Categories by Total Install Reach")
plt.xlabel("Total Installs")
plt.ylabel("Category")
plt.ticklabel_format(style="plain", axis="x")
plt.show()


**Observation:** Install reach measures market exposure. Categories with very high total installs can be attractive for products with a scalable free or freemium model.


## 7. Relationship between reviews, installs and ratings


In [ ]:
sample = df.dropna(subset=["Rating", "Reviews", "Installs"]).copy()

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=sample.sample(min(5000, len(sample)), random_state=42),
    x="Reviews", y="Installs", hue="Rating",
    palette="viridis", alpha=.55
)
plt.xscale("log")
plt.yscale("log")
plt.title("Reviews vs Installs, Coloured by Rating")
plt.xlabel("Reviews (log scale)")
plt.ylabel("Installs (log scale)")
plt.show()


**Observation:** Reviews and installs are both popularity indicators and are typically strongly related. Rating, however, does not automatically increase with install volume. A large user base can still contain mixed user experiences.


## 8. Free vs paid apps


In [ ]:
type_counts = df["Type"].value_counts()

plt.figure(figsize=(7, 5))
sns.barplot(x=type_counts.index, y=type_counts.values)
plt.title("Free vs Paid Apps")
plt.xlabel("App Type")
plt.ylabel("Number of Apps")
plt.show()

paid = df[df["Type"] == "Paid"].copy()

if len(paid):
    paid_summary = paid[["Price", "Rating", "Installs", "Reviews"]].describe().T
    display(paid_summary)


## 9. Price analysis


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=paid[paid["Price"] <= paid["Price"].quantile(.99)], x="Price", bins=40)
plt.title("Paid App Price Distribution (up to 99th percentile)")
plt.xlabel("Price")
plt.ylabel("Number of Paid Apps")
plt.show()


**Observation:** Paid-app pricing is typically concentrated toward lower price points, while a small number of expensive outliers can distort the mean. Median and percentile-based views are therefore more useful for pricing decisions.


## 10. Content rating analysis


In [ ]:
content_counts = df["Content Rating"].value_counts()

plt.figure(figsize=(9, 5))
sns.barplot(x=content_counts.values, y=content_counts.index)
plt.title("Apps by Content Rating")
plt.xlabel("Number of Apps")
plt.ylabel("Content Rating")
plt.show()


## 11. Top apps by installs and reviews


In [ ]:
top_installs = (
    df[["App", "Category", "Rating", "Installs"]]
    .sort_values("Installs", ascending=False)
    .head(10)
)

top_reviews = (
    df[["App", "Category", "Rating", "Reviews"]]
    .sort_values("Reviews", ascending=False)
    .head(10)
)

print("Top 10 apps by installs")
display(top_installs)

print("Top 10 apps by reviews")
display(top_reviews)


## 12. Business insights and recommendations

### For developers
- Entering a crowded category requires differentiation, strong UX and a clear value proposition.
- Install reach should be analysed together with ratings and reviews rather than used alone.
- Free/freemium distribution can maximise reach, while paid apps should justify their price with clear value.
- Ratings and review volume can be monitored as early indicators of product quality and user engagement.

### For app-market analysts
- Category size and total installs show market reach.
- Average rating indicates perceived quality but can be unstable for small samples.
- Reviews provide a useful engagement signal.
- Price distributions are skewed, so median/percentile statistics are useful.

### Limitations
This dataset is a historical snapshot and may not represent the current Google Play Store. Install counts are reported in broad buckets, and missing ratings/sizes limit some analyses.


In [ ]:
# Save cleaned data and key analysis outputs
df.to_csv("../output/cleaned_googleplaystore.csv", index=False)
category_rating.to_csv("../output/category_rating_summary.csv")
top_installs.to_csv("../output/top_10_installed_apps.csv", index=False)
top_reviews.to_csv("../output/top_10_reviewed_apps.csv", index=False)

summary = pd.DataFrame({
    "Metric": [
        "Raw rows",
        "Cleaned unique apps",
        "Average rating",
        "Total install reach",
        "Paid app count",
        "Free app count"
    ],
    "Value": [
        len(raw),
        len(df),
        df["Rating"].mean(),
        df["Installs"].sum(),
        int((df["Type"] == "Paid").sum()),
        int((df["Type"] == "Free").sum())
    ]
})
summary.to_csv("../output/market_summary.csv", index=False)

display(summary)
print("Android App Market analysis completed.")
